## Exploración inicial de ENCOVI 2023

Este notebook realiza un análisis exploratorio de los microdatos de ENCOVI 2023. Se cargan las bases de Personas y Hogares, se verifican las columnas principales y se revisa la estructura de los ponderadores (FACTOR / Factor2).

**Objetivo:** Generar un vistazo rápido a las dimensiones de los datos, valores faltantes y estadísticas descriptivas por departamento, sentando las bases para el cálculo de indicadores de informalidad y pobreza.

In [1]:
import pandas as pd
import pyreadstat
import os
import warnings

warnings.filterwarnings('ignore')

# Búsqueda automática de la carpeta de datos
current = os.getcwd()
data_path = None
for _ in range(5):
    candidate = os.path.join(current, 'data', 'raw', 'encovi_2023')
    if os.path.exists(candidate):
        data_path = candidate
        break
    parent = os.path.dirname(current)
    if parent == current:
        break
    current = parent

if data_path is None:
    print('No se encontró la carpeta data/raw/encovi_2023')
else:
    print('Carpeta de datos encontrada:', data_path)
    df_personas, meta = pyreadstat.read_sav(os.path.join(data_path, 'encovi_2023_personas.sav'))
    print(f'Base de Personas cargada: {df_personas.shape[0]} filas x {df_personas.shape[1]} columnas')

Carpeta de datos encontrada: C:\Users\jptor\Documents\Proyectos JP\INE-DATATON\data\raw\encovi_2023


Base de Personas cargada: 46017 filas x 626 columnas


### Muestra de datos

In [2]:
display(df_personas.head())

,DEPTO,DOMINIO,AREA,UPM,FACTOR,Factor2,NO_HOGAR,POBREZA,CP,PPA02,...,YLAB_PUBLI,Edad,Grupos_edad,PET,PEA,OCUPADOS,DESOCUPADOS,SUBVISIBLES,INACTIVOS,FORMAL_INFORMAL
0,1.0,1.0,1.0,29.0,1070.0,1070.0,4.0,3.0,1.0,1.0,...,4000.0,9.0,3.0,1.0,1.0,1.0,NaN,NaN,NaN,2.0
1,1.0,1.0,1.0,29.0,1070.0,1070.0,2.0,3.0,1.0,1.0,...,4500.0,13.0,3.0,1.0,1.0,1.0,NaN,1.0,NaN,1.0
2,1.0,1.0,1.0,29.0,1070.0,1070.0,2.0,3.0,2.0,2.0,...,NaN,18.0,4.0,1.0,NaN,NaN,NaN,NaN,1.0,NaN
3,1.0,1.0,1.0,29.0,1070.0,1070.0,1.0,3.0,1.0,2.0,...,NaN,18.0,4.0,1.0,NaN,NaN,NaN,NaN,1.0,NaN
4,1.0,1.0,1.0,29.0,1070.0,1070.0,3.0,3.0,1.0,1.0,...,6250.0,9.0,3.0,1.0,1.0,1.0,NaN,1.0,NaN,2.0


### Estadísticas descriptivas

In [3]:
df_num = df_personas.select_dtypes(include=['float64', 'int64'])
print(df_num.describe().transpose())

                   count          mean          std   min     25%      50%  \
DEPTO            46017.0     11.071169     6.501444   1.0     5.0     11.0   
DOMINIO          46017.0      2.434535     0.607867   1.0     2.0      2.0   
AREA             46017.0      1.496425     0.499993   1.0     1.0      1.0   
UPM              46017.0  14334.122520  7045.715305  29.0  8013.0  13318.0   
FACTOR           46017.0    374.580438   299.205451  22.0   179.0    276.0   
...                  ...           ...          ...   ...     ...      ...   
OCUPADOS         18036.0      1.000000     0.000000   1.0     1.0      1.0   
DESOCUPADOS        265.0      1.000000     0.000000   1.0     1.0      1.0   
SUBVISIBLES       2225.0      1.000000     0.000000   1.0     1.0      1.0   
INACTIVOS        20761.0      1.000000     0.000000   1.0     1.0      1.0   
FORMAL_INFORMAL  18036.0      1.265968     0.441860   1.0     1.0      1.0   

                     75%      max  
DEPTO               17.0   

### Verificación de ponderadores

In [4]:
print('Conteo de FACTOR no nulo:', df_personas['FACTOR'].notna().sum())
if 'Factor3' in df_personas.columns:
    print('Conteo de Factor3 no nulo:', df_personas['Factor3'].notna().sum())
    print('Correlación Pearson:', df_personas['FACTOR'].corr(df_personas['Factor3']))
elif 'Factor2' in df_personas.columns:
    print('Factor3 no existe. Usando Factor2...')
    print('Conteo de Factor2 no nulo:', df_personas['Factor2'].notna().sum())
    print('Correlación Pearson:', df_personas['FACTOR'].corr(df_personas['Factor2']))
else:
    print('Ningún factor alternativo encontrado.')

Conteo de FACTOR no nulo: 46017
Factor3 no existe. Usando Factor2...
Conteo de Factor2 no nulo: 46017
Correlación Pearson: 0.9976422736957594
